In [1]:
# Cell 1: Imports and Advanced Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score, precision_recall_curve, roc_curve
from xgboost import XGBClassifier
import shap
import pickle
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('dark_background')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn imported")
print(f"XGBoost imported")
print(f"SHAP imported")
print("Ready for Part 4 - Model Improvement!")

All libraries imported successfully!
Pandas version: 2.3.3
NumPy version: 2.4.3
Scikit-learn imported
XGBoost imported
SHAP imported
Ready for Part 4 - Model Improvement!


In [2]:
# Cell 2: Load All Datasets and Build Base Features

# Load all datasets
races = pd.read_csv('../data/races.csv')
results = pd.read_csv('../data/results.csv')
drivers = pd.read_csv('../data/drivers.csv')
constructors = pd.read_csv('../data/constructors.csv')
driver_standings = pd.read_csv('../data/driver_standings.csv')
constructor_standings = pd.read_csv('../data/constructor_standings.csv')
qualifying = pd.read_csv('../data/qualifying.csv')
pit_stops = pd.read_csv('../data/pit_stops.csv')
circuits = pd.read_csv('../data/circuits.csv')

print("All datasets loaded!")
print(f"Races: {races.shape}")
print(f"Results: {results.shape}")
print(f"Drivers: {drivers.shape}")
print(f"Qualifying: {qualifying.shape}")
print(f"Pit stops: {pit_stops.shape}")

# Filter to modern era (2020-2024)
modern_races = races[races['year'].between(2020, 2024)].copy()
modern_results = results[results['raceId'].isin(modern_races['raceId'])].copy()
modern_quali = qualifying[qualifying['raceId'].isin(modern_races['raceId'])].copy()
modern_pitstops = pit_stops[pit_stops['raceId'].isin(modern_races['raceId'])].copy()

print(f"\nFiltered to 2020-2024:")
print(f"Modern races: {modern_races.shape[0]} races")
print(f"Modern results: {modern_results.shape[0]} results")

# Merge with race info
df = modern_results.merge(modern_races[['raceId', 'year', 'round', 'circuitId', 'name']], 
                          on='raceId', how='left')

# Create target variable: is_winner (1 if finished P1, 0 otherwise)
df['is_winner'] = (df['positionOrder'] == 1).astype(int)

# Handle missing grid positions (set to last place)
df['grid'] = df['grid'].replace(0, 20)

# Base features from Part 3
df['grid_squared'] = df['grid'] ** 2
df['is_pole'] = (df['grid'] == 1).astype(int)
df['is_front_row'] = (df['grid'] <= 2).astype(int)

print(f"\nTarget variable created:")
print(f"Total winners: {df['is_winner'].sum()}")
print(f"Total non-winners: {(df['is_winner'] == 0).sum()}")
print(f"Class balance: {df['is_winner'].mean():.3f}")

print("\nBase features created: grid, grid_squared, is_pole, is_front_row")
print("Ready for advanced feature engineering!")

All datasets loaded!
Races: (1125, 18)
Results: (26759, 18)
Drivers: (861, 9)
Qualifying: (10494, 9)
Pit stops: (11371, 7)

Filtered to 2020-2024:
Modern races: 107 races
Modern results: 2139 results

Target variable created:
Total winners: 107
Total non-winners: 2032
Class balance: 0.050

Base features created: grid, grid_squared, is_pole, is_front_row
Ready for advanced feature engineering!


In [3]:
# Cell 3: Advanced Feature Engineering (Enhanced for Part 4)

print("Building advanced features...\n")

# 1. DRIVER HISTORICAL PERFORMANCE
driver_stats = df.groupby('driverId').agg({
    'points': 'mean',
    'is_winner': 'mean',
    'positionOrder': 'mean'
}).reset_index()
driver_stats.columns = ['driverId', 'driver_avg_points', 'driver_win_rate', 'driver_avg_finish']

df = df.merge(driver_stats, on='driverId', how='left')

# 2. DRIVER RECENT FORM (last 3 and 5 races)
df = df.sort_values(['driverId', 'year', 'round'])

df['driver_form_3'] = df.groupby('driverId')['points'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean().shift(1)
)
df['driver_form_5'] = df.groupby('driverId')['points'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

# 3. RECENT WIN RATE (last 10 races)
df['recent_win_rate'] = df.groupby('driverId')['is_winner'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean().shift(1)
)

# 4. CIRCUIT-SPECIFIC PERFORMANCE
circuit_driver_stats = df.groupby(['driverId', 'circuitId']).agg({
    'is_winner': 'mean',
    'points': 'mean'
}).reset_index()
circuit_driver_stats.columns = ['driverId', 'circuitId', 'win_rate_at_circuit', 'avg_points_at_circuit']

df = df.merge(circuit_driver_stats, on=['driverId', 'circuitId'], how='left')

# 5. CONSTRUCTOR PERFORMANCE
constructor_stats = df.groupby('constructorId').agg({
    'points': 'mean'
}).reset_index()
constructor_stats.columns = ['constructorId', 'constructor_avg_points']

df = df.merge(constructor_stats, on='constructorId', how='left')

# 6. CONSTRUCTOR RECENT FORM
df['constructor_form_3'] = df.groupby('constructorId')['points'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean().shift(1)
)

# 7. CHAMPIONSHIP STANDINGS (if available up to that race)
# Using driver_standings data
standings_data = driver_standings.merge(
    races[['raceId', 'year', 'round']], on='raceId', how='left'
)

# Get previous round standings
standings_data['prev_round'] = standings_data['round'] - 1
prev_standings = standings_data[['driverId', 'year', 'prev_round', 'position', 'points']].copy()
prev_standings.columns = ['driverId', 'year', 'round', 'championship_position', 'championship_points']

df = df.merge(prev_standings, on=['driverId', 'year', 'round'], how='left')

# Fill missing championship data (race 1 of season)
df['championship_position'] = df['championship_position'].fillna(20)
df['championship_points'] = df['championship_points'].fillna(0)

# 8. DNF RATE (reliability metric)
df['is_dnf'] = (df['positionOrder'] > 20).astype(int)
df['dnf_rate'] = df.groupby('driverId')['is_dnf'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean().shift(1)
)

# 9. NEW FEATURES FOR PART 4 - QUALIFYING PERFORMANCE
quali_merge = modern_quali.groupby('raceId').agg({
    'position': lambda x: x.min()
}).reset_index()

df['quali_position'] = df['grid']  # Using grid as quali position proxy

# 10. GRID POSITION IMPROVEMENT (how much driver typically gains from grid)
df['typical_gain'] = df.groupby('driverId').apply(
    lambda x: (x['grid'] - x['positionOrder']).mean()
).reset_index(name='typical_gain')['typical_gain']

df = df.merge(
    df.groupby('driverId')['typical_gain'].first().reset_index(),
    on='driverId', 
    how='left',
    suffixes=('', '_y')
)
df = df.drop(columns=['typical_gain_y'], errors='ignore')

# 11. PIT STOP EFFICIENCY
pitstop_stats = modern_pitstops.groupby(['raceId', 'driverId']).agg({
    'milliseconds': ['count', 'mean']
}).reset_index()
pitstop_stats.columns = ['raceId', 'driverId', 'num_pitstops', 'avg_pitstop_time']

df = df.merge(pitstop_stats, on=['raceId', 'driverId'], how='left')
df['num_pitstops'] = df['num_pitstops'].fillna(0)
df['avg_pitstop_time'] = df['avg_pitstop_time'].fillna(df['avg_pitstop_time'].median())

# Fill any remaining NaN values
df = df.fillna(0)

# List all features
feature_columns = [
    'grid', 'grid_squared', 'is_pole', 'is_front_row',
    'driver_avg_points', 'driver_form_3', 'driver_form_5', 
    'recent_win_rate', 'win_rate_at_circuit', 'avg_points_at_circuit',
    'constructor_avg_points', 'constructor_form_3',
    'championship_position', 'championship_points',
    'dnf_rate', 'round', 'driver_avg_finish', 'driver_win_rate',
    'typical_gain', 'num_pitstops', 'avg_pitstop_time'
]

print(f"Total features created: {len(feature_columns)}")
print("\nFeature list:")
for i, feat in enumerate(feature_columns, 1):
    print(f"{i}. {feat}")

# Split into train (2021-2023) and test (2024)
train_df = df[df['year'].between(2021, 2023)].copy()
test_df = df[df['year'] == 2024].copy()

X_train = train_df[feature_columns]
y_train = train_df['is_winner']
X_test = test_df[feature_columns]
y_test = test_df['is_winner']

print(f"\nTrain set: {X_train.shape[0]} samples ({y_train.sum()} winners)")
print(f"Test set: {X_test.shape[0]} samples ({y_test.sum()} winners)")
print(f"\nTrain class balance: {y_train.mean():.3f}")
print(f"Test class balance: {y_test.mean():.3f}")

print("\nFeature engineering complete!")

Building advanced features...

Total features created: 21

Feature list:
1. grid
2. grid_squared
3. is_pole
4. is_front_row
5. driver_avg_points
6. driver_form_3
7. driver_form_5
8. recent_win_rate
9. win_rate_at_circuit
10. avg_points_at_circuit
11. constructor_avg_points
12. constructor_form_3
13. championship_position
14. championship_points
15. dnf_rate
16. round
17. driver_avg_finish
18. driver_win_rate
19. typical_gain
20. num_pitstops
21. avg_pitstop_time

Train set: 1320 samples (66 winners)
Test set: 479 samples (24 winners)

Train class balance: 0.050
Test class balance: 0.050

Feature engineering complete!


In [4]:
# Cell 4: Hyperparameter Tuning with Cross Validation

from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, f1_score

print("Starting Hyperparameter Tuning...")
print("="*60)

# Stratified K-Fold (respects class imbalance)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Scorer - using F1 since data is imbalanced
scorer = make_scorer(f1_score, zero_division=0)

# ─────────────────────────────────────────────────────────────
# 1. XGBOOST TUNING
# ─────────────────────────────────────────────────────────────
print("\n[1/2] Tuning XGBoost...")
print("This may take 1-2 minutes...\n")

xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'scale_pos_weight': [10, 15, 20]   # handles class imbalance
}

xgb_base = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

xgb_grid = GridSearchCV(
    estimator=xgb_base,
    param_grid=xgb_param_grid,
    scoring=scorer,
    cv=cv,
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)

best_xgb = xgb_grid.best_estimator_
print(f"\nBest XGBoost Parameters:")
for param, value in xgb_grid.best_params_.items():
    print(f"  {param}: {value}")
print(f"Best CV F1 Score: {xgb_grid.best_score_:.4f}")

# ─────────────────────────────────────────────────────────────
# 2. RANDOM FOREST TUNING
# ─────────────────────────────────────────────────────────────
print("\n[2/2] Tuning Random Forest...")
print("This may take 1-2 minutes...\n")

rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample']
}

rf_base = RandomForestClassifier(random_state=42)

rf_grid = GridSearchCV(
    estimator=rf_base,
    param_grid=rf_param_grid,
    scoring=scorer,
    cv=cv,
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
print(f"\nBest Random Forest Parameters:")
for param, value in rf_grid.best_params_.items():
    print(f"  {param}: {value}")
print(f"Best CV F1 Score: {rf_grid.best_score_:.4f}")

# ─────────────────────────────────────────────────────────────
# 3. CROSS VALIDATION SCORES - BOTH MODELS
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("Cross Validation Results (5-Fold Stratified)")
print("="*60)

xgb_cv_scores = cross_val_score(best_xgb, X_train, y_train, cv=cv, scoring=scorer)
rf_cv_scores  = cross_val_score(best_rf,  X_train, y_train, cv=cv, scoring=scorer)

print(f"\nXGBoost  F1 per fold: {[round(s,4) for s in xgb_cv_scores]}")
print(f"XGBoost  Mean F1: {xgb_cv_scores.mean():.4f}  (+/- {xgb_cv_scores.std():.4f})")

print(f"\nRand Forest F1 per fold: {[round(s,4) for s in rf_cv_scores]}")
print(f"Rand Forest Mean F1: {rf_cv_scores.mean():.4f}  (+/- {rf_cv_scores.std():.4f})")

# ─────────────────────────────────────────────────────────────
# 4. EVALUATE ON TEST SET
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("Test Set Performance (2024 Season)")
print("="*60)

# XGBoost predictions
xgb_pred  = best_xgb.predict(X_test)
xgb_proba = best_xgb.predict_proba(X_test)[:, 1]

# Random Forest predictions
rf_pred  = best_rf.predict(X_test)
rf_proba = best_rf.predict_proba(X_test)[:, 1]

# Metrics
for model_name, pred, proba in [
    ("XGBoost",       xgb_pred, xgb_proba),
    ("Random Forest", rf_pred,  rf_proba)
]:
    acc = accuracy_score(y_test, pred)
    f1  = f1_score(y_test, pred, zero_division=0)
    auc = roc_auc_score(y_test, proba)
    print(f"\n{model_name}:")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1 Score : {f1:.4f}")
    print(f"  ROC-AUC  : {auc:.4f}")
    print(f"  Classification Report:")
    print(classification_report(y_test, pred, zero_division=0))

# Determine best model
if f1_score(y_test, xgb_pred) >= f1_score(y_test, rf_pred):
    best_model      = best_xgb
    best_model_name = "XGBoost"
else:
    best_model      = best_rf
    best_model_name = "Random Forest"

print(f"\nBest overall model: {best_model_name}")
print("\nHyperparameter tuning complete!")

Starting Hyperparameter Tuning...

[1/2] Tuning XGBoost...
This may take 1-2 minutes...

Fitting 5 folds for each of 324 candidates, totalling 1620 fits

Best XGBoost Parameters:
  colsample_bytree: 1.0
  learning_rate: 0.05
  max_depth: 3
  n_estimators: 200
  scale_pos_weight: 15
  subsample: 1.0
Best CV F1 Score: 0.7819

[2/2] Tuning Random Forest...
This may take 1-2 minutes...

Fitting 5 folds for each of 216 candidates, totalling 1080 fits

Best Random Forest Parameters:
  class_weight: balanced_subsample
  max_depth: 6
  min_samples_leaf: 2
  min_samples_split: 5
  n_estimators: 100
Best CV F1 Score: 0.7622

Cross Validation Results (5-Fold Stratified)

XGBoost  F1 per fold: [np.float64(0.8125), np.float64(0.7857), np.float64(0.8387), np.float64(0.8667), np.float64(0.6061)]
XGBoost  Mean F1: 0.7819  (+/- 0.0920)

Rand Forest F1 per fold: [np.float64(0.8148), np.float64(0.7857), np.float64(0.7857), np.float64(0.8), np.float64(0.625)]
Rand Forest Mean F1: 0.7622  (+/- 0.0695)

Tes